# Multilingual QA Evaluation Showcase

This notebook is a cleaner portfolio-style companion to the original training notebook. It focuses on:

- loading the saved multilingual QA checkpoint
- showing a corrected preprocessing function for future fine-tuning
- evaluating the model with Exact Match (EM) and token-level F1
- breaking down results by language on a lightweight sample

It does **not** retrain the model, so it is much easier to run on a laptop.

In [ ]:
%pip install -q transformers datasets pandas tqdm accelerate

## 1. Imports and Configuration

Update `MODEL_PATH` to wherever your trained checkpoint is stored.

Suggested options:
- `./TheModel/checkpoint-7500` if you copy the checkpoint next to this notebook
- your Linux path if you are running Jupyter from WSL
- your Windows path if you are running Jupyter from Windows

In [ ]:
from pathlib import Path
import re
import string
from collections import Counter, defaultdict

import pandas as pd
import torch
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline

MODEL_PATH = r"./TheModel/checkpoint-7500"
BASE_TOKENIZER = "xlm-roberta-base"
MAX_LENGTH = 448
EVAL_SAMPLES = 100
SEED = 42

device = 0 if torch.cuda.is_available() else -1
print("device:", "cuda" if device == 0 else "cpu")
print("model path:", MODEL_PATH)
print("path exists:", Path(MODEL_PATH).exists())

## 2. Load the Dataset

We keep the original TyDi QA dataset. If a built-in validation split exists, we use it. Otherwise, we create a reproducible holdout split from the training set.

In [ ]:
ds = load_dataset("google-research-datasets/tydiqa", "secondary_task")
print(ds)

if "validation" in ds:
    eval_dataset = ds["validation"]
    split_name = "validation"
else:
    split = ds["train"].train_test_split(test_size=0.1, seed=SEED)
    eval_dataset = split["test"]
    split_name = "train/test holdout"

languages = [example_id.split("-")[0] for example_id in ds["train"]["id"]]
lang_counts = Counter(languages)

print("evaluation split:", split_name)
print("train language counts:")
pd.DataFrame(sorted(lang_counts.items()), columns=["language", "count"])

## 3. Corrected Preprocessing Function

This cell is here to show stronger ML understanding. It fixes the main issue from the original notebook by aligning the answer only against the **context tokens**, not the whole question-context pair.

You do not need to run training again right now, but this is the version you should keep for any future fine-tuning.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_TOKENIZER)

def preprocess_example(example):
    tokenized = tokenizer(
        example["question"],
        example["context"],
        truncation="only_second",
        padding="max_length",
        max_length=MAX_LENGTH,
        return_offsets_mapping=True
    )

    offsets = tokenized["offset_mapping"]
    sequence_ids = tokenized.sequence_ids()

    answer_text = example["answers"]["text"][0]
    answer_start = example["answers"]["answer_start"][0]
    answer_end = answer_start + len(answer_text)

    context_token_indices = [i for i, s in enumerate(sequence_ids) if s == 1]

    start_position = 0
    end_position = 0

    for idx in context_token_indices:
        start_char, end_char = offsets[idx]
        if start_char <= answer_start < end_char:
            start_position = idx
        if start_char < answer_end <= end_char:
            end_position = idx

    tokenized["start_positions"] = start_position
    tokenized["end_positions"] = end_position
    tokenized.pop("offset_mapping")
    return tokenized

sample_features = preprocess_example(ds["train"][0])
print(sample_features.keys())

## 4. Load the Saved Model

This section loads the already trained checkpoint and builds a QA inference pipeline. No retraining happens here.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_TOKENIZER)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_PATH)

qa_pipeline = pipeline(
    task="question-answering",
    model=model,
    tokenizer=tokenizer,
    device=device
)

print("checkpoint loaded successfully")

## 5. Define EM and F1 Metrics

These are simple, readable implementations that are good for a portfolio notebook. They are lightweight and easy to explain in interviews.

In [ ]:
def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = "".join(ch for ch in text if ch not in string.punctuation)
    return text

def exact_match_score(prediction, ground_truth):
    return int(normalize_text(prediction) == normalize_text(ground_truth))

def f1_score_tokens(prediction, ground_truth):
    pred_tokens = normalize_text(prediction).split()
    gold_tokens = normalize_text(ground_truth).split()

    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    common = Counter(pred_tokens) & Counter(gold_tokens)
    overlap = sum(common.values())

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def best_metric_over_references(metric_fn, prediction, references):
    return max(metric_fn(prediction, ref) for ref in references)

def extract_language(example_id):
    return example_id.split("-")[0]

## 6. Run a Lightweight Evaluation

To keep the notebook friendly for a laptop, we evaluate on a sample instead of the entire split. You can increase `EVAL_SAMPLES` later if you want a stronger benchmark.

In [ ]:
eval_size = min(EVAL_SAMPLES, len(eval_dataset))
sampled_eval = eval_dataset.shuffle(seed=SEED).select(range(eval_size))

records = []

for example in tqdm(sampled_eval, total=len(sampled_eval)):
    prediction = qa_pipeline(question=example["question"], context=example["context"])
    predicted_answer = prediction["answer"]
    gold_answers = [text for text in example["answers"]["text"] if text.strip()]

    em = best_metric_over_references(exact_match_score, predicted_answer, gold_answers)
    f1 = best_metric_over_references(f1_score_tokens, predicted_answer, gold_answers)

    records.append({
        "id": example["id"],
        "language": extract_language(example["id"]),
        "question": example["question"],
        "prediction": predicted_answer,
        "gold_answers": gold_answers,
        "em": em,
        "f1": f1,
        "score": prediction.get("score")
    })

results_df = pd.DataFrame(records)
results_df.head()

In [ ]:
overall_em = results_df["em"].mean() * 100
overall_f1 = results_df["f1"].mean() * 100

summary_df = pd.DataFrame([
    {"metric": "Exact Match", "value": round(overall_em, 2)},
    {"metric": "F1", "value": round(overall_f1, 2)},
    {"metric": "Evaluated samples", "value": len(results_df)}
])

summary_df

In [ ]:
language_breakdown = (
    results_df.groupby("language")[["em", "f1"]]
    .mean()
    .mul(100)
    .round(2)
    .sort_values("f1", ascending=False)
    .reset_index()
)

language_breakdown

## 7. Inspect Good and Bad Examples

This is useful for interviews and LinkedIn because it shows that you do not stop at training. You also inspect the model's strengths and failure cases.

In [ ]:
best_examples = results_df.sort_values(["f1", "score"], ascending=[False, False]).head(5)
worst_examples = results_df.sort_values(["f1", "score"], ascending=[True, True]).head(5)

print("Best examples")
display(best_examples[["language", "question", "prediction", "gold_answers", "em", "f1", "score"]])

print("Worst examples")
display(worst_examples[["language", "question", "prediction", "gold_answers", "em", "f1", "score"]])

## 8. Optional Demo Questions

Use this section for quick screenshots after the evaluation is done.

In [ ]:
demo_context = """Paris is the capital of France. القاهرة هي عاصمة مصر. Berlin is in Germany."""

demo_questions = [
    "What is the capital of France?",
    "ما هي عاصمة مصر؟",
    "What city is in Germany?"
]

for q in demo_questions:
    pred = qa_pipeline(question=q, context=demo_context)
    print(f"Q: {q}")
    print(f"A: {pred['answer']}")
    print(f"Score: {pred['score']:.4f}")
    print("-" * 60)